# Silver layer

# Preprocessing for text cleaning is as follows:
- Normalize line endings
- Replace tabs and non-breaking spaces
- Remove zero-width spaces if present
- Fix hyphenated line breaks
- Remove trailing spaces on lines
- Remove repeated spaces inside lines
- Normalize spaces around line breaks
- Merge broken PDF lines where useful for reading
- Remove common PDF page markers
- Remove reference-like lines
- Remove duplicate lines where useful for reading
- Reduce too many blank lines

# We create two cleaned text versions:
- cleaned_text: reading-friendly text for summarization / general downstream use
- nlp_text: structure-preserving text for NLP / entity extraction / front-matter logic

Input: bronze JSON
Output: silver JSON with cleaned text fields

In [25]:
# Import needed libraries
import pandas as pd
import re
from pathlib import Path
from typing import Any, Dict, List, Tuple

In [26]:
# Paths
INPUT_PATH = Path("../../Data/bronze/doc_01_bronze.json")
OUTPUT_PATH = Path("../../Data/silver/doc_01_silver.json")

## Load bronze dataframe + sanity checks

In [27]:
# Load bronze data
project_data = pd.read_json(INPUT_PATH, orient="records")

# Show preview
print(project_data.head())

# Basic validation
required_columns = ["raw_text"]
missing_columns = [col for col in required_columns if col not in project_data.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"\nLoaded {len(project_data)} rows from {INPUT_PATH.name}")
print("\nColumns:")
print(project_data.columns.tolist())

print("\nNull counts:")
print(project_data[required_columns].isnull().sum())

  document_id                                           raw_text
0      doc_01  De Zeeuwse arbeidsmarkt\nAanbod en vraag in Ze...

Loaded 1 rows from doc_01_bronze.json

Columns:
['document_id', 'raw_text']

Null counts:
raw_text    0
dtype: int64


# Clean incoming text data

In [28]:
def safe_string(text):
    """Return a safe string for downstream cleaning."""
    if pd.isna(text):
        return ""
    return str(text)

First we will Normalize line endings

In [29]:
def normalize_line_endings(text):
    text = safe_string(text)
    return text.replace("\r\n", "\n").replace("\r", "\n")
#   print(normalize_line_endings(project_data['raw_text']))

Replace tabs and non-breaking spaces

In [30]:
def replace_special_spaces(text):
    text = safe_string(text)
    text = text.replace("\t", " ")
    text = text.replace("\xa0", " ")
    return text

Remove zero-width spaces if present

In [31]:
def remove_zero_width_characters(text):
    text = safe_string(text)
    zero_width_chars = ['\u200B', '\u200C', '\u200D', '\uFEFF']
    for char in zero_width_chars:
        text = text.replace(char, '')
    return text

Remove trailing spaces on lines

In [32]:
def fix_hyphenated_linebreaks(text):
    text = safe_string(text)
    # Example: "infor-\nmation" -> "information"
    return re.sub(r'(\w)-\n(\w)', r'\1\2', text)

Remove repeated spaces inside lines

In [33]:
def remove_pdf_page_markers(text):
    text = safe_string(text)

    patterns = [
        r'Page\s+\d+\s+of\s+\d+',
        r'^\s*Page\s+\d+\s*$',
        r'^\s*\d+\s*\|\s*P\s*a\s*g\s*e\s*$',
        r'^\s*P\s*a\s*g\s*e\s*\d+\s*$',
    ]

    for pattern in patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.MULTILINE)

    return text

Normalize spaces around line breaks

In [34]:
def remove_urls_and_emails(text):
    text = safe_string(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', '', text)
    return text

Reduce too many blank lines

In [35]:
def remove_trailing_spaces(text):
    text = safe_string(text)
    return re.sub(r'[ \t]+$', '', text, flags=re.MULTILINE)

Remove very common PDF page markers if present

In [36]:
def remove_repeated_spaces(text):
    text = safe_string(text)
    return re.sub(r' {2,}', ' ', text)

In [37]:
def normalize_linebreak_spacing(text):
    text = safe_string(text)
    return re.sub(r' *\n *', '\n', text)

In [38]:
def normalize_linebreak_spacing_soft(text):
    text = safe_string(text)
    return re.sub(r'[ \t]*\n[ \t]*', '\n', text)

In [39]:
def merge_broken_lines(text):
    text = safe_string(text)
    lines = text.split('\n')
    merged = []

    for line in lines:
        line = line.strip()

        if not line:
            merged.append("")
            continue

        if not merged:
            merged.append(line)
            continue

        prev = merged[-1]

        # Never merge if the previous line is short (likely a heading or front matter)
        if len(prev.split()) <= 8:
            merged.append(line)
            continue

        # Never merge if the current line is short and standalone-looking
        if len(line.split()) <= 4 and line[0].isupper():
            merged.append(line)
            continue

        # Keep list items / numbered lines as separate lines
        if re.match(r'^(\d+[\.\)]\s+|[-•]\s+)', line):
            merged.append(line)
            continue

        # Merge only if previous line clearly continues (no sentence-ending punctuation)
        # and current line is a lowercase continuation
        if not prev.endswith(('.', '!', '?', ':')) and line[0].islower():
            merged[-1] = prev + ' ' + line
        else:
            merged.append(line)

    return '\n'.join(merged)

In [40]:
def remove_reference_like_lines(text):
    text = safe_string(text)
    cleaned_lines = []

    for line in text.split('\n'):
        stripped = line.strip()

        if not stripped:
            cleaned_lines.append("")
            continue

        # bibliography-like patterns
        if re.match(r'^[A-Z][a-zA-Z\-]+,\s+[A-Z]\.', stripped):
            continue
        if re.search(r'\(\d{4}\)', stripped) and len(stripped.split()) < 12:
            continue
        if stripped.lower().startswith("references"):
            continue
        if stripped.lower().startswith("bibliography"):
            continue

        cleaned_lines.append(line)

    return '\n'.join(cleaned_lines)

In [41]:
def remove_duplicate_lines(text):
    text = safe_string(text)
    seen = set()
    result = []

    for line in text.split('\n'):
        key = re.sub(r'\s+', ' ', line.strip().lower())

        if key and key in seen:
            continue
        if key:
            seen.add(key)

        result.append(line)

    return '\n'.join(result)

In [42]:
def remove_many_blank_lines(text):
    text = safe_string(text)
    return re.sub(r'\n{3,}', '\n\n', text)

Run all functions on text

In [43]:
def clean_text_for_reading(text):
    text = normalize_line_endings(text)
    text = replace_special_spaces(text)
    text = remove_zero_width_characters(text)
    text = fix_hyphenated_linebreaks(text)
    text = remove_pdf_page_markers(text)
    text = remove_urls_and_emails(text)
    text = remove_trailing_spaces(text)
    text = remove_repeated_spaces(text)
    text = normalize_linebreak_spacing(text)
    text = merge_broken_lines(text)
    text = remove_reference_like_lines(text)
    text = remove_duplicate_lines(text)
    text = remove_many_blank_lines(text)
    return text.strip()


In [44]:
# NLP-friendly cleaning pipeline
def clean_text_for_nlp(text):
    text = normalize_line_endings(text)
    text = replace_special_spaces(text)
    text = remove_zero_width_characters(text)
    text = fix_hyphenated_linebreaks(text)
    text = remove_pdf_page_markers(text)
    text = remove_urls_and_emails(text)
    text = remove_trailing_spaces(text)
    text = remove_repeated_spaces(text)
    text = normalize_linebreak_spacing_soft(text)
    text = remove_reference_like_lines(text)
    text = remove_many_blank_lines(text)
    return text.strip()

Run sanity checks

In [45]:
# Sanity checks
def sanity_check(text, n_chars=700):
    original = safe_string(text)
    cleaned_reading = clean_text_for_reading(original)
    cleaned_nlp = clean_text_for_nlp(original)

    print("Original Text:\n")
    print(original[:n_chars])

    print("\n" + "-" * 60 + "\n")
    print("Cleaned Text for Reading:\n")
    print(cleaned_reading[:n_chars])

    print("\n" + "-" * 60 + "\n")
    print("Cleaned Text for NLP:\n")
    print(cleaned_nlp[:n_chars])

    return cleaned_reading, cleaned_nlp


_ = sanity_check(project_data.loc[0, "raw_text"])

Original Text:

De Zeeuwse arbeidsmarkt
Aanbod en vraag in Zeeland
Periodieke publicatie van de huidige stand van zaken, de 
knelpunten en een prognose naar de toekomst
November 2018
Els van der Reep
Sven Bakker Uitgave: #1

Inhoudsopgave
PROGNOSE – PAGINA 32  
VERWACHT TEKORT IN 2022: 6.000
In 2022 is er naar verwachting een tekort aan 
6.000 werkende personen. Dit zijn banen die 
vanuit de arbeidsreserve, van buiten de 
provincie of door een groei in het aantal uren 
per werkende moeten worden opgevuld.
KNELPUNTEN – PAGINA 26
Inkomende pendelaars: 24.800 personen
Uitgaande pendelaars: 41.900 personen
ARBEIDSVRAAG – PAGINA 14  ARBEIDSAANBOD – PAGINA 2
Aantal banen: 180.000
Aantal banen 2017-2012: -1.76

------------------------------------------------------------

Cleaned Text for Reading:

De Zeeuwse arbeidsmarkt
Aanbod en vraag in Zeeland
Periodieke publicatie van de huidige stand van zaken, de knelpunten en een prognose naar de toekomst
November 2018
Els van der Reep
Sven Bakke

In [46]:
project_data["cleaned_text"] = project_data["raw_text"].apply(clean_text_for_reading)
project_data["nlp_text"] = project_data["raw_text"].apply(clean_text_for_nlp)

print(project_data[["raw_text", "cleaned_text", "nlp_text"]].head(1))

                                            raw_text  \
0  De Zeeuwse arbeidsmarkt\nAanbod en vraag in Ze...   

                                        cleaned_text  \
0  De Zeeuwse arbeidsmarkt\nAanbod en vraag in Ze...   

                                            nlp_text  
0  De Zeeuwse arbeidsmarkt\nAanbod en vraag in Ze...  


In [47]:
# Quality checks
project_data["raw_length"] = project_data["raw_text"].apply(lambda x: len(safe_string(x)))
project_data["cleaned_length"] = project_data["cleaned_text"].apply(len)
project_data["nlp_length"] = project_data["nlp_text"].apply(len)

project_data["cleaned_ratio"] = project_data["cleaned_length"] / project_data["raw_length"].replace(0, 1)
project_data["nlp_ratio"] = project_data["nlp_length"] / project_data["raw_length"].replace(0, 1)

print(project_data[["raw_length", "cleaned_length", "nlp_length", "cleaned_ratio", "nlp_ratio"]].describe())

print("\nRows where cleaned text is very short:")
print(project_data.loc[project_data["cleaned_length"] < 200, ["cleaned_length", "nlp_length"]].head())



       raw_length  cleaned_length  nlp_length  cleaned_ratio  nlp_ratio
count         1.0             1.0         1.0       1.000000   1.000000
mean      44231.0         41171.0     43096.0       0.930818   0.974339
std           NaN             NaN         NaN            NaN        NaN
min       44231.0         41171.0     43096.0       0.930818   0.974339
25%       44231.0         41171.0     43096.0       0.930818   0.974339
50%       44231.0         41171.0     43096.0       0.930818   0.974339
75%       44231.0         41171.0     43096.0       0.930818   0.974339
max       44231.0         41171.0     43096.0       0.930818   0.974339

Rows where cleaned text is very short:
Empty DataFrame
Columns: [cleaned_length, nlp_length]
Index: []


In [48]:
# Save cleaned data
project_data.to_json(OUTPUT_PATH, orient="records", indent=2)

print(f"Saved cleaned silver data to: {OUTPUT_PATH}")

Saved cleaned silver data to: ..\..\Data\silver\doc_01_silver.json
